# UbiFusionNet v1 — same-data publication experiment

This notebook trains the proposed architecture on the exact released MMUbiPred training samples. Training and model selection do not access the locked independent test set. Use a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
if not (REPO_ROOT / '.git').exists():
    subprocess.run([
        'git', 'clone', '--recurse-submodules',
        'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
        str(REPO_ROOT),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True)

os.chdir(REPO_ROOT)
print('Project:', REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt

In [ ]:
import platform
import sys
import numpy as np
import sklearn
import torch

print('Python:', sys.version)
print('Platform:', platform.platform())
print('NumPy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Preflight tests

In [ ]:
subprocess.run([
    sys.executable, '-m', 'unittest', 'discover',
    '-s', 'experiment/tests',
], check=True)

## Audit immutable benchmark artifacts

In [ ]:
subprocess.run([
    sys.executable, 'experiment/scripts/audit_dataset.py',
    '--data-dir', 'replication/MMUbiPred',
    '--output', 'experiment/outputs/dataset_audit.json',
], check=True)

## Select on development data, then refit on the complete released training set

In [ ]:
subprocess.run([
    sys.executable, 'experiment/scripts/train.py',
    '--config', 'experiment/configs/ubifusion_v1.json',
], check=True)

In [ ]:
import json

RUN_DIR = REPO_ROOT / 'experiment/outputs/ubifusion_v1_seed42'
validation_metrics = json.loads((RUN_DIR / 'validation_metrics.json').read_text())
validation_metrics

## Locked independent-test evaluation

Leave `RUN_LOCKED_TEST=False` during architecture development. Set it to `True` only after this configuration has been accepted from validation results.

In [ ]:
RUN_LOCKED_TEST = False

if RUN_LOCKED_TEST:
    subprocess.run([
        sys.executable, 'experiment/scripts/evaluate.py',
        '--run-dir', str(RUN_DIR),
        '--data-dir', 'replication/MMUbiPred',
        '--allow-locked-test',
    ], check=True)
    locked_metrics = json.loads((RUN_DIR / 'locked_test_metrics.json').read_text())
    display(locked_metrics)
else:
    print('Locked test was not accessed.')

## Preserve the run in Google Drive

In [ ]:
import shutil
from datetime import date

DRIVE_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments') / f'{date.today().isoformat()}-ubifusion-v1-seed42'
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
for artifact in RUN_DIR.iterdir():
    destination = DRIVE_RUN_DIR / artifact.name
    if artifact.is_dir():
        shutil.copytree(artifact, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(artifact, destination)
print('Saved run artifacts to:', DRIVE_RUN_DIR)